# cALorie — locked six-creator region benchmark v1

This notebook performs the one-time frozen **V6-C** versus **V6-F** evaluation
on the locked 36-region creator benchmark. It performs **no training,
calibration, threshold selection, relabeling, or checkpoint modification**.

The contract contains all 14 supported exercises plus four compound `unknown`
regions from six reserved creator videos. Its scope is region classification,
not continuous long-video segmentation. Run all cells; no upload is required.
The final cell downloads `calorie_locked_creator_region_benchmark_v1_report.zip`.


In [ ]:
from pathlib import Path
import hashlib, json, math, os, random, shutil, urllib.request, zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, confusion_matrix,
    f1_score, precision_score, recall_score,
)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
WORK = Path("/content/calorie_locked_creator_region_benchmark_v1")
DOWNLOADS = WORK / "downloads"
EVIDENCE = WORK / "evidence"
REPORT = WORK / "report"
for path in (DOWNLOADS, EVIDENCE, REPORT): path.mkdir(parents=True, exist_ok=True)

SOURCE_COMMIT = "27bc6898a9e466e9b36be0c0c9af94e0b5b5ceab"
RAW_ROOT = f"https://raw.githubusercontent.com/bariscelikk1/cALorie/{SOURCE_COMMIT}"
WINDOW, STRIDE = 64, 32
TARGET_FPS = 15.0
SUPPORTED_CLASSES = [
    "bench_press", "bicep_curl", "deadlift", "dumbbell_row",
    "jumping_jack", "lat_pulldown", "lateral_raise", "lunge",
    "pull_up", "push_up", "shoulder_press", "sit_up", "squat",
    "tricep_extension",
]
ALL_CLASSES = SUPPORTED_CLASSES + ["unknown"]
V6C_THRESHOLD = 0.675
V6F_THRESHOLD = 0.65
print("Device:", DEVICE)
print("Pinned source commit:", SOURCE_COMMIT)


## 1. Download and verify immutable inputs

The notebook fetches only pose evidence and frozen checkpoints from a pinned
Git commit. Raw YouTube video is neither requested nor exported.


In [ ]:
FILES = {
    "evidence.zip": (
        "ml/artifacts/youtube_creator_benchmark_v1/evidence/calorie_creator_benchmark_v1_evidence.zip",
        "3c9930fc3086b68240082668648722836c8959c373a58268347788d7ab24b3c4",
    ),
    "gapfill_evidence.zip": (
        "ml/artifacts/youtube_creator_benchmark_v1/gapfill_evidence/calorie_creator_benchmark_v1_gapfill_evidence.zip",
        "4d0f3efed5216f0f46e96478c9777fc652814a346046f5866309357efc38e050",
    ),
    "v6c.pt": (
        "ml/artifacts/v6c/model/two_stage_tcn_v6c.pt",
        "ec3cf00dccb77ab52f04374b3f4d1b4c012d8894999bd453e1034a7a33ae95b2",
    ),
    "v6f_classifier.pt": (
        "ml/artifacts/v6f_multistream_gate/model/frozen_temporal_classifier_checkpoint.pt",
        "c30ecd625d77f31380a503f881eff5c540b8a7cdbbb71a39276c33b908947b92",
    ),
    "joint_gate.pt": (
        "ml/artifacts/v6f_multistream_gate/model/joint_gate.pt",
        "a01b0f9c00bcd69150fbd10db385fddb5e3c8b35497acae620dc1fa35f8f6cfb",
    ),
    "bone_gate.pt": (
        "ml/artifacts/v6f_multistream_gate/model/bone_gate.pt",
        "ef7d3e19a7c237e99c7cad956b6f39e13f5aa8a36ff965c167ee690f4d400707",
    ),
    "velocity_gate.pt": (
        "ml/artifacts/v6f_multistream_gate/model/velocity_gate.pt",
        "3af308a69145981ca0a5ea79b243b3d0b04d18f7c75ff49c59a088e9ce50788c",
    ),
    "locked_contract.csv": (
        "ml/data/youtube_creator_benchmark_v1/locked_region_benchmark_v1.csv",
        "86bb7780b17e7872e00fc427eed5de6267845b2b9a99e85e260486e039ff041d",
    ),
    "lock_audit.json": (
        "ml/data/youtube_creator_benchmark_v1/locked_region_benchmark_audit_v1.json",
        "20d2005ba4ffda1e2a7a1f7d4880df2b4894d70756950fafc086d4b0d38c56d2",
    ),
}

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def download_verified(name, relative_path, expected):
    target = DOWNLOADS / name
    if not target.exists() or sha256(target) != expected:
        target.unlink(missing_ok=True)
        print("Downloading", name)
        urllib.request.urlretrieve(f"{RAW_ROOT}/{relative_path}", target)
    actual = sha256(target)
    assert actual == expected, (name, actual, expected)
    print(f"verified {name}: {actual[:12]}…")
    return target

downloaded = {
    name: download_verified(name, path, digest)
    for name, (path, digest) in FILES.items()
}

def safe_extract(archive, destination):
    destination = destination.resolve()
    with zipfile.ZipFile(archive) as zf:
        for member in zf.infolist():
            target = (destination / member.filename).resolve()
            assert target == destination or destination in target.parents, member.filename
        zf.extractall(destination)

safe_extract(downloaded["evidence.zip"], EVIDENCE / "base")
safe_extract(downloaded["gapfill_evidence.zip"], EVIDENCE / "gapfill")
print("All immutable inputs verified.")


## 2. Verify the locked region contract

The CSV and its lock audit are checksum-pinned. The notebook refuses to score
unless the contract is locked, contains exactly 36 regions from six videos,
covers all 15 output labels, and preserves every fail-closed prohibition.


In [ ]:
lock_audit = json.loads(downloaded["lock_audit.json"].read_text())
assert lock_audit["region_benchmark_locked"] is True
assert lock_audit["full_timeline_benchmark_locked"] is False
assert lock_audit["errors"] == []

segments = pd.read_csv(downloaded["locked_contract.csv"]).rename(
    columns={"region_id": "interval_id"}
)
assert len(segments) == 36
assert segments.video_id.nunique() == 6
assert segments.creator.nunique() == 6
assert set(segments.label) == set(ALL_CLASSES)
assert (segments.label == "unknown").sum() == 4
assert segments.interval_id.is_unique
assert segments.lock_status.eq("locked_region_classification_v1").all()
assert segments.benchmark_scope.eq("region_classification_only").all()
for field in (
    "use_for_training", "calibration_allowed", "threshold_selection_allowed",
    "predictions_used_for_labeling",
):
    assert segments[field].astype(str).str.lower().eq("false").all(), field

segments.to_csv(REPORT / "locked_region_benchmark_v1.csv", index=False)
display(segments.groupby(["creator", "label"]).size().unstack(fill_value=0))
print(
    "Locked regions:", len(segments),
    "| supported:", (segments.label != "unknown").sum(),
    "| unknown:", (segments.label == "unknown").sum(),
)


## 3. Build deterministic 64-frame windows

Windows are extracted only from the locked region boundaries. The final short
window is included deterministically; sparse missing pose frames are linearly
interpolated exactly as in the verified Notebook 35 reproduction.


In [ ]:
VIDEO_FILES = {}
for root in (EVIDENCE / "base", EVIDENCE / "gapfill"):
    for path in root.glob("continuous_landmarks/*.npz"):
        VIDEO_FILES[path.stem] = path

assert set(segments.video_id).issubset(VIDEO_FILES), (set(segments.video_id) - set(VIDEO_FILES))

def interpolate_pose(pose, detected):
    pose = pose.astype(np.float32, copy=True)
    detected = detected.astype(bool)
    good = np.flatnonzero(detected & np.isfinite(pose).all(axis=(1, 2)))
    if len(good) == 0:
        raise ValueError("region has no detected finite pose")
    x = np.arange(len(pose))
    flat = pose.reshape(len(pose), -1)
    for column in range(flat.shape[1]):
        flat[:, column] = np.interp(x, good, flat[good, column])
    return flat.reshape(pose.shape).astype(np.float32)

def resample_to_window(pose):
    if len(pose) == WINDOW: return pose.astype(np.float32)
    old = np.arange(len(pose), dtype=np.float32)
    new = np.linspace(0, len(pose) - 1, WINDOW, dtype=np.float32)
    flat = pose.reshape(len(pose), -1)
    out = np.stack([np.interp(new, old, flat[:, j]) for j in range(flat.shape[1])], axis=1)
    return out.reshape(WINDOW, 17, 4).astype(np.float32)

window_arrays, window_rows, region_audit = [], [], []
for row in segments.itertuples(index=False):
    with np.load(VIDEO_FILES[row.video_id], allow_pickle=False) as data:
        pose = data["landmarks"]
        timestamps = data["timestamps_s"]
        detected = data["detected"]
    mask = (timestamps >= row.start_s) & (timestamps <= row.end_s)
    assert mask.any(), row.interval_id
    clip = pose[mask]
    clip_detected = detected[mask]
    detection_rate = float(clip_detected.mean())
    clip = interpolate_pose(clip, clip_detected)
    if len(clip) < WINDOW:
        starts = [0]
        windows = [resample_to_window(clip)]
    else:
        starts = list(range(0, len(clip) - WINDOW + 1, STRIDE))
        final_start = len(clip) - WINDOW
        if starts[-1] != final_start: starts.append(final_start)
        windows = [clip[start:start + WINDOW].astype(np.float32) for start in starts]
    region_audit.append({
        "interval_id": row.interval_id, "source_frames": len(clip),
        "window_count": len(windows), "pose_detection_rate": detection_rate,
    })
    for local_index, (start, window) in enumerate(zip(starts, windows)):
        assert window.shape == (WINDOW, 17, 4)
        assert np.isfinite(window).all()
        window_arrays.append(window)
        window_rows.append({
            "interval_id": row.interval_id, "video_id": row.video_id,
            "creator": row.creator, "true_label": row.label,
            "window_index": local_index,
            "window_start_frame_in_region": int(start),
        })

X = np.stack(window_arrays).astype(np.float32)
window_meta = pd.DataFrame(window_rows)
region_audit = pd.DataFrame(region_audit)
region_audit.to_csv(REPORT / "region_pose_audit.csv", index=False)
print("Window tensor:", X.shape)
display(region_audit.describe(include="all"))


## 4. Load the frozen V6-C and V6-F systems


In [ ]:
BONE_EDGES = [
    (5, 6), (5, 7), (7, 9), (6, 8), (8, 10), (5, 11),
    (6, 12), (11, 12), (11, 13), (13, 15), (12, 14), (14, 16),
]
ANGLE_TRIPLETS = [
    (5, 7, 9), (6, 8, 10), (7, 5, 11), (8, 6, 12),
    (5, 11, 13), (6, 12, 14), (11, 13, 15), (12, 14, 16),
]

def pose_to_features(sample):
    xy = sample[:, :, :2].astype(np.float32)
    velocity = np.zeros_like(xy); velocity[1:] = xy[1:] - xy[:-1]
    bones = np.stack([xy[:, child] - xy[:, parent] for parent, child in BONE_EDGES], axis=1)
    angles = []
    for first, center, last in ANGLE_TRIPLETS:
        a = xy[:, first] - xy[:, center]
        b = xy[:, last] - xy[:, center]
        denominator = np.linalg.norm(a, axis=1) * np.linalg.norm(b, axis=1)
        cosine = np.sum(a * b, axis=1) / np.maximum(denominator, 1e-6)
        angles.append(np.arccos(np.clip(cosine, -1.0, 1.0)) / np.pi)
    angles = np.stack(angles, axis=1).astype(np.float32)
    features = np.concatenate([
        xy.reshape(WINDOW, -1), velocity.reshape(WINDOW, -1),
        bones.reshape(WINDOW, -1), angles,
    ], axis=1).astype(np.float32)
    assert features.shape == (WINDOW, 100)
    return features

def pose_to_stream(sample, stream):
    xy = sample[:, :, :2].astype(np.float32)
    if stream == "joint": return xy.reshape(WINDOW, -1)
    if stream == "velocity":
        velocity = np.zeros_like(xy); velocity[1:] = xy[1:] - xy[:-1]
        return velocity.reshape(WINDOW, -1)
    if stream == "bone":
        return np.stack([xy[:, child] - xy[:, parent] for parent, child in BONE_EDGES], axis=1).reshape(WINDOW, -1)
    raise ValueError(stream)

class TemporalBlock(nn.Module):
    def __init__(self, in_channels, out_channels, dilation, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(in_channels, out_channels, 3, padding=dilation, dilation=dilation),
            nn.BatchNorm1d(out_channels), nn.ReLU(), nn.Dropout(dropout),
            nn.Conv1d(out_channels, out_channels, 3, padding=dilation, dilation=dilation),
            nn.BatchNorm1d(out_channels), nn.ReLU(), nn.Dropout(dropout),
        )
        self.residual = nn.Conv1d(in_channels, out_channels, 1) if in_channels != out_channels else nn.Identity()
    def forward(self, x): return torch.relu(self.net(x) + self.residual(x))

class SmallTCN(nn.Module):
    def __init__(self, input_features, num_classes):
        super().__init__()
        channels = [64, 96, 128, 128]
        blocks, current = [], input_features
        for level, output in enumerate(channels):
            blocks.append(TemporalBlock(current, output, dilation=2 ** level))
            current = output
        self.features = nn.Sequential(*blocks)
        self.classifier = nn.Linear(current, num_classes)
    def forward(self, x): return self.classifier(self.features(x).mean(dim=-1))

v6c_checkpoint = torch.load(downloaded["v6c.pt"], map_location=DEVICE, weights_only=False)
assert list(v6c_checkpoint["supported_classes"]) == SUPPORTED_CLASSES
v6c_gate = SmallTCN(100, 2).to(DEVICE)
v6c_exercise = SmallTCN(100, len(SUPPORTED_CLASSES)).to(DEVICE)
v6c_gate.load_state_dict(v6c_checkpoint["gate_state_dict"])
v6c_exercise.load_state_dict(v6c_checkpoint["exercise_state_dict"])

v6f_classifier_checkpoint = torch.load(downloaded["v6f_classifier.pt"], map_location=DEVICE, weights_only=False)
v6f_exercise = SmallTCN(100, len(SUPPORTED_CLASSES)).to(DEVICE)
v6f_exercise.load_state_dict(v6f_classifier_checkpoint["exercise_state_dict"])
stream_dims = {"joint": 34, "bone": 24, "velocity": 34}
v6f_gates = {}
for stream, dimension in stream_dims.items():
    model = SmallTCN(dimension, 2).to(DEVICE)
    gate_checkpoint = torch.load(
        downloaded[f"{stream}_gate.pt"], map_location=DEVICE, weights_only=False
    )
    assert gate_checkpoint["stream"] == stream
    assert int(gate_checkpoint["input_features"]) == dimension
    model.load_state_dict(gate_checkpoint["state_dict"])
    v6f_gates[stream] = model
for model in [v6c_gate, v6c_exercise, v6f_exercise, *v6f_gates.values()]: model.eval()
print("Frozen models loaded.")


## 5. Run inference and aggregate windows to locked regions


In [ ]:
def batched_probabilities(model, arrays, transform, batch_size=256):
    output = []
    with torch.no_grad():
        for start in range(0, len(arrays), batch_size):
            batch = np.stack([transform(x) for x in arrays[start:start + batch_size]]).transpose(0, 2, 1)
            logits = model(torch.from_numpy(batch).to(DEVICE))
            output.append(logits.softmax(1).cpu().numpy())
    return np.concatenate(output)

features = lambda x: pose_to_features(x)
v6c_gate_prob = batched_probabilities(v6c_gate, X, features)[:, 1]
v6c_exercise_prob = batched_probabilities(v6c_exercise, X, features)
v6f_exercise_prob = batched_probabilities(v6f_exercise, X, features)
v6f_stream_prob = {
    stream: batched_probabilities(model, X, lambda x, s=stream: pose_to_stream(x, s))[:, 1]
    for stream, model in v6f_gates.items()
}
v6f_gate_prob = np.stack([v6f_stream_prob[s] for s in ("joint", "bone", "velocity")], axis=1).mean(axis=1)

for prefix, gate_prob, exercise_prob, threshold in [
    ("v6c", v6c_gate_prob, v6c_exercise_prob, V6C_THRESHOLD),
    ("v6f", v6f_gate_prob, v6f_exercise_prob, V6F_THRESHOLD),
]:
    exercise_index = exercise_prob.argmax(axis=1)
    window_meta[f"{prefix}_gate_supported_probability"] = gate_prob
    window_meta[f"{prefix}_exercise_prediction"] = [SUPPORTED_CLASSES[i] for i in exercise_index]
    window_meta[f"{prefix}_prediction"] = np.where(
        gate_prob >= threshold,
        np.asarray([SUPPORTED_CLASSES[i] for i in exercise_index]),
        "unknown",
    )
    for class_index, class_name in enumerate(SUPPORTED_CLASSES):
        window_meta[f"{prefix}_exercise_probability__{class_name}"] = exercise_prob[:, class_index]

window_meta.to_csv(REPORT / "window_predictions.csv", index=False)

region_predictions = []
for interval_id, group in window_meta.groupby("interval_id", sort=False):
    base = segments.loc[segments.interval_id == interval_id].iloc[0]
    record = {
        "interval_id": interval_id, "video_id": base.video_id,
        "creator": base.creator, "true_label": base.label,
        "window_count": len(group),
    }
    for prefix, threshold in [("v6c", V6C_THRESHOLD), ("v6f", V6F_THRESHOLD)]:
        gate = float(group[f"{prefix}_gate_supported_probability"].mean())
        probabilities = np.asarray([
            group[f"{prefix}_exercise_probability__{name}"].mean()
            for name in SUPPORTED_CLASSES
        ])
        class_name = SUPPORTED_CLASSES[int(probabilities.argmax())]
        record[f"{prefix}_gate_supported_probability"] = gate
        record[f"{prefix}_exercise_prediction"] = class_name
        record[f"{prefix}_prediction"] = class_name if gate >= threshold else "unknown"
    region_predictions.append(record)

region_predictions = pd.DataFrame(region_predictions)
region_predictions.to_csv(REPORT / "region_predictions.csv", index=False)
display(region_predictions)


## 6. Frozen metrics and creator-bootstrap uncertainty

All 15 labels are present. The creator bootstrap resamples the six reserved
creators as units and computes paired V6-F minus V6-C intervals. These
intervals quantify source-level sampling uncertainty; they are not permission
to tune thresholds or rerun variants on this benchmark.


In [ ]:
def metric_row(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    present = sorted(set(y_true), key=ALL_CLASSES.index)
    supported = y_true != "unknown"
    true_unknown = y_true == "unknown"
    pred_unknown = y_pred == "unknown"
    gate_true = np.where(true_unknown, "unknown", "supported")
    gate_pred = np.where(pred_unknown, "unknown", "supported")
    gate_recalls = [
        float((gate_pred[gate_true == label] == label).mean())
        for label in ("unknown", "supported") if (gate_true == label).any()
    ]
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "full_15_label_macro_f1": float(f1_score(y_true, y_pred, labels=ALL_CLASSES, average="macro", zero_division=0)),
        "present_label_macro_f1": float(f1_score(y_true, y_pred, labels=present, average="macro", zero_division=0)),
        "supported_accuracy": float(accuracy_score(y_true[supported], y_pred[supported])) if supported.any() else None,
        "unknown_precision": float(precision_score(true_unknown, pred_unknown, zero_division=0)),
        "unknown_recall": float(recall_score(true_unknown, pred_unknown, zero_division=0)),
        "gate_balanced_accuracy": float(np.mean(gate_recalls)),
        "regions": int(len(y_true)),
        "present_labels": present,
    }

y_true = region_predictions.true_label.to_numpy()
metrics = {}
for system in ("v6c", "v6f"):
    metrics[system] = metric_row(y_true, region_predictions[f"{system}_prediction"].to_numpy())

metric_table = pd.DataFrame(metrics).T.reset_index(names="system")
metric_table.to_csv(REPORT / "region_metrics.csv", index=False)
display(metric_table)

class_rows = []
for system in ("v6c", "v6f"):
    pred = region_predictions[f"{system}_prediction"].to_numpy()
    for label in ALL_CLASSES:
        support = int((y_true == label).sum())
        class_rows.append({
            "system": system, "label": label, "support": support,
            "precision": float(precision_score(y_true, pred, labels=[label], average="macro", zero_division=0)),
            "recall": float(recall_score(y_true, pred, labels=[label], average="macro", zero_division=0)),
            "f1": float(f1_score(y_true, pred, labels=[label], average="macro", zero_division=0)),
        })
class_metrics = pd.DataFrame(class_rows)
class_metrics.to_csv(REPORT / "class_metrics.csv", index=False)

creator_rows = []
for creator, group in region_predictions.groupby("creator"):
    for system in ("v6c", "v6f"):
        row = metric_row(group.true_label, group[f"{system}_prediction"])
        row.update({"creator": creator, "system": system})
        creator_rows.append(row)
creator_metrics = pd.DataFrame(creator_rows)
creator_metrics.to_csv(REPORT / "creator_metrics.csv", index=False)

BOOTSTRAPS = 2000
rng = np.random.default_rng(SEED)
creators = region_predictions.creator.unique()
bootstrap_rows = []
for iteration in range(BOOTSTRAPS):
    sampled = rng.choice(creators, size=len(creators), replace=True)
    sample = pd.concat([region_predictions[region_predictions.creator == c] for c in sampled], ignore_index=True)
    for system in ("v6c", "v6f"):
        row = metric_row(sample.true_label, sample[f"{system}_prediction"])
        bootstrap_rows.append({"iteration": iteration, "system": system, **{k: row[k] for k in ["accuracy", "full_15_label_macro_f1", "present_label_macro_f1", "supported_accuracy", "unknown_precision", "unknown_recall", "gate_balanced_accuracy"]}})
bootstrap = pd.DataFrame(bootstrap_rows)
bootstrap.to_csv(REPORT / "creator_bootstrap_draws.csv", index=False)

ci_rows = []
numeric_metrics = ["accuracy", "full_15_label_macro_f1", "present_label_macro_f1", "supported_accuracy", "unknown_precision", "unknown_recall", "gate_balanced_accuracy"]
for metric in numeric_metrics:
    for system in ("v6c", "v6f"):
        values = bootstrap.loc[bootstrap.system == system, metric].dropna()
        ci_rows.append({"comparison": system, "metric": metric, "estimate": metrics[system][metric], "ci_2_5": float(values.quantile(.025)), "ci_97_5": float(values.quantile(.975))})
    pivot = bootstrap.pivot(index="iteration", columns="system", values=metric).dropna()
    delta = pivot.v6f - pivot.v6c
    ci_rows.append({"comparison": "v6f_minus_v6c", "metric": metric, "estimate": metrics["v6f"][metric] - metrics["v6c"][metric], "ci_2_5": float(delta.quantile(.025)), "ci_97_5": float(delta.quantile(.975))})
bootstrap_ci = pd.DataFrame(ci_rows)
bootstrap_ci.to_csv(REPORT / "creator_bootstrap_ci.csv", index=False)
display(bootstrap_ci)


In [ ]:
for system in ("v6c", "v6f"):
    matrix = confusion_matrix(y_true, region_predictions[f"{system}_prediction"], labels=ALL_CLASSES)
    fig, ax = plt.subplots(figsize=(12, 10))
    image = ax.imshow(matrix, cmap="Blues")
    ax.set_xticks(range(len(ALL_CLASSES)), ALL_CLASSES, rotation=90)
    ax.set_yticks(range(len(ALL_CLASSES)), ALL_CLASSES)
    ax.set_xlabel("Predicted"); ax.set_ylabel("Locked region label")
    ax.set_title(f"{system.upper()} locked region confusion matrix")
    for i in range(len(ALL_CLASSES)):
        for j in range(len(ALL_CLASSES)):
            if matrix[i, j]: ax.text(j, i, str(matrix[i, j]), ha="center", va="center", fontsize=8)
    fig.colorbar(image, ax=ax); fig.tight_layout()
    fig.savefig(REPORT / f"{system}_region_confusion_matrix.png", dpi=180)
    plt.close(fig)


## 7. Export the immutable benchmark report

The report records the pinned contract, checkpoints, fixed thresholds, metrics,
predictions, source-level bootstrap intervals, and checksums. It does not make
an automatic promotion decision and must never be used for threshold tuning.


In [ ]:
run_metadata = {
    "experiment": "locked_six_creator_region_benchmark_v1",
    "source_commit": SOURCE_COMMIT,
    "seed": SEED,
    "window_frames": WINDOW,
    "stride_frames": STRIDE,
    "target_fps": TARGET_FPS,
    "systems": {
        "v6c": {
            "gate_threshold": V6C_THRESHOLD,
            "checkpoint_sha256": FILES["v6c.pt"][1],
        },
        "v6f": {
            "fusion": "mean_probability",
            "gate_threshold": V6F_THRESHOLD,
            "classifier_checkpoint_sha256": FILES["v6f_classifier.pt"][1],
            "gate_checkpoint_sha256": {
                stream: FILES[f"{stream}_gate.pt"][1]
                for stream in ("joint", "bone", "velocity")
            },
        },
    },
    "contract": {
        "sha256": FILES["locked_contract.csv"][1],
        "lock_audit_sha256": FILES["lock_audit.json"][1],
        "regions": int(len(segments)),
        "supported_regions": int((segments.label != "unknown").sum()),
        "unknown_regions": int((segments.label == "unknown").sum()),
        "creators": int(segments.creator.nunique()),
        "labels": ALL_CLASSES,
        "lock_status": "locked_region_classification_v1",
        "benchmark_scope": "region_classification_only",
        "full_timeline_locked": False,
        "eligible_for_training": False,
        "eligible_for_calibration": False,
        "eligible_for_threshold_selection": False,
        "prediction_guided_relabeling_allowed": False,
    },
    "primary_metrics": metrics,
    "interpretation": (
        "One-time frozen region-classification benchmark. "
        "No continuous long-video segmentation claim and no tuning allowed."
    ),
}
(REPORT / "metrics.json").write_text(json.dumps(run_metadata, indent=2) + "\n")

readme = (
    "# cALorie locked creator region benchmark v1\n\n"
    "This report compares frozen V6-C and V6-F on 36 locked regions from six "
    "reserved creator videos. All 14 supported labels plus unknown are present. "
    "No training, calibration, threshold selection, or relabeling occurred. "
    "The scope is region classification, not continuous long-video segmentation. "
    "See metrics.json, locked_region_benchmark_v1.csv, and region_predictions.csv.\n"
)
(REPORT / "README.md").write_text(readme)

checksums = []
for path in sorted(REPORT.iterdir()):
    if path.is_file() and path.name != "SHA256SUMS.txt":
        checksums.append(f"{sha256(path)}  {path.name}")
(REPORT / "SHA256SUMS.txt").write_text("\n".join(checksums) + "\n")

archive_base = WORK / "calorie_locked_creator_region_benchmark_v1_report"
archive = Path(shutil.make_archive(str(archive_base), "zip", root_dir=REPORT))
print("Report:", archive)
print("SHA-256:", sha256(archive))
print("\nPRIMARY RESULT")
display(metric_table)
try:
    from google.colab import files
    files.download(str(archive))
except Exception:
    print("Download manually from:", archive)
